# 02 · Validate results and compare disconnection effects

Use the same profile and run ID as notebook 01. This stage checks case provenance, replays forecast selection and seed-level metrics, and regenerates the joint decisions from the completed run. It exports figures and a standalone interactive view.


In [ ]:
import sys
from pathlib import Path

if sys.version_info[:2] != (3, 14):
    raise RuntimeError("Select the project Python 3.14 .venv kernel.")

working_directory = Path.cwd().resolve()
try:
    REPO = next(
        path
        for path in (working_directory, *working_directory.parents)
        if (path / "pyproject.toml").is_file() and (path / "src" / "effectome").is_dir()
    )
except StopIteration as error:
    raise RuntimeError("Open this notebook from inside the effectome application repository.") from error

project_src = str(REPO / "src")
if project_src not in sys.path:
    sys.path.insert(0, project_src)

print("Notebook kernel", sys.executable)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
PROJECT_ROOT = REPO


In [ ]:
from pathlib import Path

ROOT = PROJECT_ROOT
PROFILE = "confirmation"  # Set "smoke" here and in the other run notebook for a short check.
RUN_ID = "support-01"

if PROFILE not in {"smoke", "confirmation"}:
    raise ValueError("PROFILE must be smoke or confirmation.")
if not RUN_ID or Path(RUN_ID).name != RUN_ID or RUN_ID in {".", ".."}:
    raise ValueError("Choose a single directory name for RUN_ID.")
RUN_ROOT = ROOT / "outputs" / "effectome-rollout-support" / PROFILE / RUN_ID
print("Run directory", RUN_ROOT)


In [ ]:
import os

import pandas as pd
from IPython.display import FileLink, Image, display

from synthetic.support_validation import validate_and_visualize

result = validate_and_visualize(RUN_ROOT)
display({key: result[key] for key in ("profile", "case_count", "seed_count", "support", "learned_selector")})
metrics = pd.read_csv(result["seed_metrics"])
columns = ["forecast_mae", "early_forecast_mae", "active_effect_mae"]
display(metrics.groupby(["scenario", "arm"])[columns].mean().round(5))
for title, files in result["figures"].items():
    print(title)
    display(Image(filename=files["png"]))
    display(FileLink(os.path.relpath(files["pdf"], Path.cwd())))
print("Open the paired intervention viewer in a browser", result["interactive_view"])
display(FileLink(os.path.relpath(result["interactive_view"], Path.cwd())))


The browser view lets you choose a condition, seed, fitted arm, held-out origin, and saved neuron or group. It compares signed fitted effects with the simulator on both behavioral coordinates. A smoke run reports descriptive diagnostics on one seed.
